# Test du vote inter-colonnes — Déf. 6.1–6.2

Ce notebook remplace le script Python et teste le mécanisme de vote distal entre colonnes corticales.

Objectifs :
- vérifier la convergence basique du consensus,
- mesurer l'effet du nombre de colonnes sur le nombre d'étapes jusqu'à stabilité,
- vérifier la résolution d'ambiguïté par intersection.

Exécuter les cellules dans l'ordre.

In [1]:
from pathlib import Path
import sys
from typing import List, Sequence, Tuple

import torch


def resolve_project_root() -> Path:
    """Retrouve la racine du dépôt à partir du notebook."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "cortical_column.py").exists():
            return candidate
    raise FileNotFoundError("Impossible de localiser cortical_column.py")


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cortical_column import CorticalColumn, SDRSpace, MultiColumnConsensus


torch.manual_seed(42)


def make_input_pattern(n_in: int, active: int = 64, seed: int = 7) -> torch.BoolTensor:
    """Construit une entrée binaire stable pour le SpatialPooler."""
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(n_in, generator=generator)[:active]
    x = torch.zeros(n_in, dtype=torch.bool)
    x[indices] = True
    return x


def make_columns(
    n_columns: int,
    seed: int = 42,
    n_in: int = 1024,
    n_mc: int = 512,
) -> list[CorticalColumn]:
    """Crée N colonnes avec les mêmes poids initiaux et des orientations initiales différentes."""
    torch.manual_seed(seed)
    columns = [
        CorticalColumn(
            N_in=n_in,
            N_mc=n_mc,
            sdr_w=40,
            s=0.02,
            d=2,
            K_grid=4,
            K_columns=n_columns,
        )
        for _ in range(n_columns)
    ]

    for idx, column in enumerate(columns):
        # On initialise des orientations distinctes pour simuler des points de vue différents.
        column.l6b.theta_idx = torch.tensor((idx * 73) % column.l6b.N, dtype=torch.long)

    return columns


def make_noise_blocks(
    object_sdr: torch.BoolTensor,
    n_columns: int,
    max_steps: int,
    block_size: int,
    seed: int,
) -> list[list[torch.LongTensor]]:
    """Découpe un pool de bits parasites en blocs disjoints par colonne."""
    noise_indices = torch.where(~object_sdr)[0]
    blocks_per_column = min(4, max(1, max_steps - 1))
    required = n_columns * blocks_per_column * block_size
    if required > len(noise_indices):
        raise ValueError("Pool de bruit insuffisant pour construire le scénario de test")

    generator = torch.Generator().manual_seed(seed)
    shuffled = noise_indices[torch.randperm(len(noise_indices), generator=generator)]

    schedules: list[list[torch.LongTensor]] = []
    cursor = 0
    for _ in range(n_columns):
        column_blocks: list[torch.LongTensor] = []
        for _ in range(blocks_per_column):
            block = shuffled[cursor : cursor + block_size]
            column_blocks.append(block)
            cursor += block_size
        schedules.append(column_blocks)

    return schedules


def make_view_from_core(
    n_mc: int,
    core_indices: torch.LongTensor,
    noise_block: torch.LongTensor,
) -> torch.BoolTensor:
    """Construit une hypothèse locale de taille contrôlée."""
    view = torch.zeros(n_mc, dtype=torch.bool)
    view[core_indices] = True
    view[noise_block] = True
    return view


def run_column_vote_sequence(
    columns: list[CorticalColumn],
    object_sdr: torch.BoolTensor,
    core_size: int,
    max_steps: int,
    seed: int,
    input_active: int = 64,
) -> tuple[list[torch.BoolTensor], int, torch.BoolTensor, torch.LongTensor]:
    """Exécute une séquence de présentations et renvoie la trajectoire du consensus.

    Déf. 6.1 : chaque colonne accumule une union locale de ses hypothèses.
    Déf. 6.2 : le consensus global est l'intersection des unions locales.
    """
    n_mc = object_sdr.numel()
    w = int(object_sdr.sum().item())
    core_generator = torch.Generator().manual_seed(seed + 101)

    object_indices = torch.where(object_sdr)[0]
    core_indices = object_indices[torch.randperm(len(object_indices), generator=core_generator)[:core_size]]
    block_size = w - core_size

    schedules = make_noise_blocks(
        object_sdr=object_sdr,
        n_columns=len(columns),
        max_steps=max_steps,
        block_size=block_size,
        seed=seed + 202,
    )

    histories: list[list[torch.BoolTensor]] = [[] for _ in columns]
    trajectory: list[torch.BoolTensor] = []
    base_input = make_input_pattern(columns[0].spatial_pooler.N_in, active=input_active, seed=seed + 303)
    v_ego = torch.tensor([0.1, 0.05], dtype=torch.float32)

    consensus_module = MultiColumnConsensus(n=n_mc, K=len(columns))

    for step in range(max_steps):
        unions: list[torch.BoolTensor] = []
        for col_idx, column in enumerate(columns):
            # Les blocs parasites sont disjoints entre colonnes et changent dans le temps.
            block_idx = min(step, len(schedules[col_idx]) - 1)
            noise_block = schedules[col_idx][block_idx]
            current_view = make_view_from_core(n_mc=n_mc, core_indices=core_indices, noise_block=noise_block)
            histories[col_idx].append(current_view)

            delta_orientation = (step + 1) * (col_idx + 1)
            with torch.no_grad():
                _, _, union_h = column.step(
                    I_raw=base_input,
                    v_ego=v_ego,
                    delta_orientation=delta_orientation,
                    local_hypotheses=histories[col_idx],
                    learn=False,
                )
            unions.append(union_h)

        consensus = consensus_module.consensus(unions)
        trajectory.append(consensus)

    stable_step = len(trajectory)
    for idx in range(1, len(trajectory)):
        if torch.equal(trajectory[idx], trajectory[idx - 1]):
            stable_step = idx + 1
            break

    return trajectory, stable_step, core_indices, base_input


def build_overlap_sdr(
    sdr_space: SDRSpace,
    reference: torch.BoolTensor,
    overlap: int,
    seed: int,
) -> torch.BoolTensor:
    """Construit un SDR avec un overlap contrôlé par rapport à une référence."""
    generator = torch.Generator().manual_seed(seed)
    ref_indices = torch.where(reference)[0]
    keep = ref_indices[torch.randperm(len(ref_indices), generator=generator)[:overlap]]

    non_ref_indices = torch.where(~reference)[0]
    add = non_ref_indices[torch.randperm(len(non_ref_indices), generator=generator)[: sdr_space.w - overlap]]

    sdr = torch.zeros(sdr_space.n, dtype=torch.bool)
    sdr[keep] = True
    sdr[add] = True
    return sdr


def consensus_size_report(trajectory: Sequence[torch.BoolTensor]) -> list[int]:
    return [int(step.sum().item()) for step in trajectory]


## Test 1 — Convergence basique

On instancie 4 colonnes, on leur donne le même objet, mais avec des orientations initiales différentes. Le consensus doit devenir non vide et rester stable en moins de 10 présentations.

In [2]:
def test_basic_convergence() -> None:
    """Déf. 6.1–6.2 : convergence basique du vote inter-colonnes."""
    print("\n" + "═" * 78)
    print("TEST 1 — Convergence basique")
    print("═" * 78)

    n_columns = 4
    max_steps = 10
    n_mc = 512
    w = 40
    core_size = 32

    sdr_space = SDRSpace(n=n_mc, w=w)
    object_3 = sdr_space.random_sdr()
    columns = make_columns(n_columns=n_columns, seed=123, n_in=1024, n_mc=n_mc)

    trajectory, stable_step, core_indices, _ = run_column_vote_sequence(
        columns=columns,
        object_sdr=object_3,
        core_size=core_size,
        max_steps=max_steps,
        seed=1001,
    )

    sizes = consensus_size_report(trajectory)
    print(f"Nombre de colonnes : {n_columns}")
    print(f"Objet simulant le MNIST '3' : {int(object_3.sum().item())} bits actifs")
    print(f"Core commun injecté : {len(core_indices)} bits")
    print(f"Tailles du consensus par étape : {sizes}")
    print(f"Étape de stabilité : {stable_step} / {max_steps}")

    assert sizes[-1] > 0, "Le consensus final ne doit pas être vide."
    assert stable_step <= max_steps, "Le consensus doit se stabiliser en au plus max_steps présentations."
    if stable_step < len(trajectory):
        assert torch.equal(trajectory[stable_step - 1], trajectory[stable_step - 2]), (
            "Le consensus doit être identique sur deux étapes consécutives au moment de la stabilité."
        )

    print("✓ Le consensus est non vide et stable.")
    print("✓ Déf. 6.1–6.2 vérifiées sur 4 colonnes.")


test_basic_convergence()



══════════════════════════════════════════════════════════════════════════════
TEST 1 — Convergence basique
══════════════════════════════════════════════════════════════════════════════
Nombre de colonnes : 4
Objet simulant le MNIST '3' : 40 bits actifs
Core commun injecté : 32 bits
Tailles du consensus par étape : [32, 32, 32, 32, 32, 32, 32, 32, 32, 32]
Étape de stabilité : 2 / 10
✓ Le consensus est non vide et stable.
✓ Déf. 6.1–6.2 vérifiées sur 4 colonnes.


## Test 2 — Scaling du nombre de colonnes

On mesure le nombre d'étapes nécessaires pour atteindre la stabilité pour `N ∈ {1, 2, 4, 8}`. La table affichée doit montrer que le nombre d'étapes décroît ou reste stable quand `N` augmente.

In [3]:
def test_scaling(n_values: Sequence[int] = (1, 2, 4, 8)) -> list[tuple[int, int, int]]:
    """Déf. 6.1–6.2 : mesure du temps de convergence en fonction du nombre de colonnes."""
    print("\n" + "═" * 78)
    print("TEST 2 — Scaling du nombre de colonnes")
    print("═" * 78)

    max_steps = 10
    n_mc = 512
    w = 40
    core_size = 32

    sdr_space = SDRSpace(n=n_mc, w=w)
    object_3 = sdr_space.random_sdr()

    results: list[tuple[int, int, int]] = []
    previous_steps: int | None = None

    for n_columns in n_values:
        columns = make_columns(n_columns=n_columns, seed=321, n_in=1024, n_mc=n_mc)
        trajectory, stable_step, _, _ = run_column_vote_sequence(
            columns=columns,
            object_sdr=object_3,
            core_size=core_size,
            max_steps=max_steps,
            seed=2002 + n_columns,
        )
        consensus = trajectory[stable_step - 1]
        size = int(consensus.sum().item())
        results.append((n_columns, stable_step, size))

        print(f"N={n_columns:>2} colonnes -> étapes={stable_step:>2}, consensus={size:>3} bits")

        if previous_steps is not None:
            assert stable_step <= previous_steps, (
                "Le nombre d'étapes jusqu'au consensus doit décroître ou rester stable quand N augmente."
            )
        previous_steps = stable_step

    print("\n" + "─" * 68)
    header_steps = "Étapes jusqu'au consensus"
    header_size = "Taille du consensus (bits)"
    print(f"{'N colonnes':>12} | {header_steps:>26} | {header_size:>27}")
    print("─" * 68)
    for n_columns, stable_step, size in results:
        print(f"{n_columns:>12} | {stable_step:>26} | {size:>27}")
    print("─" * 68)

    assert results[0][1] >= results[-1][1], "Le scaling doit être non croissant en fonction de N."
    print("✓ Le nombre d'étapes décroît ou reste stable quand N augmente.")

    return results


scaling_results = test_scaling()



══════════════════════════════════════════════════════════════════════════════
TEST 2 — Scaling du nombre de colonnes
══════════════════════════════════════════════════════════════════════════════
N= 1 colonnes -> étapes= 5, consensus= 74 bits
N= 2 colonnes -> étapes= 3, consensus= 33 bits
N= 4 colonnes -> étapes= 2, consensus= 32 bits
N= 8 colonnes -> étapes= 2, consensus= 32 bits

────────────────────────────────────────────────────────────────────
  N colonnes |  Étapes jusqu'au consensus |  Taille du consensus (bits)
────────────────────────────────────────────────────────────────────
           1 |                          5 |                          74
           2 |                          3 |                          33
           4 |                          2 |                          32
           8 |                          2 |                          32
────────────────────────────────────────────────────────────────────
✓ Le nombre d'étapes décroît ou reste stable q

## Test 3 — Robustesse à l'ambiguïté

On crée deux SDRs `A` et `B` avec un overlap contrôlé de 15 bits sur `w=40`, puis on présente `A` à la moitié des colonnes et `B` à l'autre moitié. Le consensus final doit être plus petit que chaque hypothèse individuelle.

In [4]:
def test_ambiguity_resolution() -> None:
    """Déf. 6.1–6.2 : l'intersection élimine l'ambiguïté."""
    print("\n" + "═" * 78)
    print("TEST 3 — Robustesse à l'ambiguïté")
    print("═" * 78)

    n_columns = 8
    n_mc = 512
    w = 40
    overlap_target = 15

    sdr_space = SDRSpace(n=n_mc, w=w)
    reference = sdr_space.random_sdr()
    sdr_a = reference
    sdr_b = build_overlap_sdr(sdr_space=sdr_space, reference=sdr_a, overlap=overlap_target, seed=909)

    overlap = int(sdr_space.overlap(sdr_a, sdr_b).item())
    half = n_columns // 2
    unions = [sdr_a.clone() for _ in range(half)] + [sdr_b.clone() for _ in range(n_columns - half)]
    consensus_module = MultiColumnConsensus(n=n_mc, K=n_columns)
    consensus = consensus_module.consensus(unions)
    consensus_size = int(consensus.sum().item())

    print(f"A : {int(sdr_a.sum().item())} bits")
    print(f"B : {int(sdr_b.sum().item())} bits")
    print(f"Overlap A ∩ B : {overlap} bits")
    print(f"Consensus final : {consensus_size} bits")

    assert abs(overlap - overlap_target) <= 0, "L'overlap doit être exactement celui demandé." 
    assert consensus_size < int(sdr_a.sum().item()), "Le consensus doit être plus petit que A." 
    assert consensus_size < int(sdr_b.sum().item()), "Le consensus doit être plus petit que B." 
    assert consensus_size <= overlap, "Le consensus doit rester au plus égal à l'intersection A ∩ B." 

    print("✓ Le consensus final est plus petit que chaque hypothèse individuelle.")
    print("✓ L'ambiguïté est bien réduite par l'intersection.")


test_ambiguity_resolution()



══════════════════════════════════════════════════════════════════════════════
TEST 3 — Robustesse à l'ambiguïté
══════════════════════════════════════════════════════════════════════════════
A : 40 bits
B : 40 bits
Overlap A ∩ B : 15 bits
Consensus final : 15 bits
✓ Le consensus final est plus petit que chaque hypothèse individuelle.
✓ L'ambiguïté est bien réduite par l'intersection.


## Exécution complète

Le notebook ci-dessus peut aussi être lancé de manière séquentielle via les trois fonctions ci-dessous.

In [ ]:
# Cellule de synthèse optionnelle.
# Les trois tests sont déjà exécutés dans leurs sections respectives.
